# Resíduos e Ljung-Box consolidado

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
from statsmodels.stats.diagnostic import acorr_ljungbox

mae = pd.read_csv(require_file(
    config.RESULT_DIR / "mae_consolidado.csv", "01_MAE_consolidado.ipynb"
))
rows = []
for model in mae["model"]:
    pred = pd.read_csv(require_file(
        config.RESULT_DIR / f"residual_diagnostic_{model}.csv",
        f"02_modelos/{model}/02_walkforward_validacao.ipynb",
    ))
    residual = pred["y_true"] - pred["y_pred"]
    lb = acorr_ljungbox(residual, lags=[1, 6, 12, 24], return_df=True)
    rows.append({
        "model": model,
        "bias": residual.mean(),
        "acf_lag1": residual.autocorr(1),
        **{
            f"p_ljung_box_{lag}": lb.loc[lag, "lb_pvalue"]
            for lag in (1, 6, 12, 24)
        },
    })
residual_summary = pd.DataFrame(rows)
residual_summary.to_csv(
    config.RESULT_DIR / "residual_diagnostics.csv", index=False
)
display(residual_summary)
print("p < 0,05 indica autocorrelação remanescente nos erros.")
print("Os lags acima são horários: o diagnóstico usa 168 origens consecutivas.")


,model,bias,acf_lag1,p_ljung_box_1,p_ljung_box_6,p_ljung_box_12,p_ljung_box_24
0,Random_Forest,42.523816,0.504723,4.871108e-11,6.409615e-09,4.738903e-12,1.454046e-11
1,PLS_Regression,-23.984250,-0.008119,9.155867e-01,2.034680e-01,4.661770e-02,6.111705e-02
2,Holt_Winters,-44.244048,0.768022,1.336966e-23,4.042205e-53,6.435862e-51,7.370339e-47
3,SARIMAX,50.798367,0.048447,5.278384e-01,2.524586e-01,1.153435e-02,2.599919e-02


p < 0,05 indica autocorrelação remanescente nos erros.
Os lags acima são horários: o diagnóstico usa 168 origens consecutivas.
